In [1]:
#pandas for spreadsheet processing
import pandas as pd
pd.set_option('display.max_columns', None) #display all columns

#numpy for mathematical functions
import numpy as np

#geopandas to process geospatial/gis data
import geopandas as gpd

#library for connecting to the geodatabase
from sqlalchemy import create_engine, text
from geoalchemy2 import Geometry

# User input

- All human input values will need to be input in by the user into the following cells. 
- The rest of the code will run without required human intervention.

## 1. Input Link to downloaded datasets

In [2]:
# highest level of qualification by sex
# To download original dataset go to - 
# https://fingertips.phe.org.uk/
income_excel = r"N:\Geodatabase\Raw_Data\Income\Income estimates for small areas.xlsx"


## 2. Input Link to MSOA 2011 shapefile
#### To download original dataset go to - 
https://geoportal.statistics.gov.uk/datasets/ons::middle-layer-super-output-areas-december-2011-boundaries-ew-bfe-v3/about

In [8]:
msoa2011_shapefile = r"N:\Geodatabase\Raw_Data\Census 2021\Boundaries\Middle_Layer_Super_Output_Areas_Dec_2011_Boundaries_Full_Extent_BFE_EW_V3_2022_46367968394682200\MSOA_2011_EW_BFE_V3.shp"

## 1. Import & Process Data

To download original dataset go to - 
https://www.nomisweb.co.uk/query/construct/summary.asp?mode=construct&version=0&dataset=2221

In [4]:
# Define sheet names
sheet_names = [
    'Total annual income',
    'Net annual income',
    'Net income before housing costs',
    'Net income after housing costs'
]

# Load each sheet into a dictionary of DataFrames, skipping the first 4 rows
data = {
    sheet: pd.read_excel(income_excel, sheet_name=sheet, skiprows=4)
    for sheet in sheet_names
}



In [6]:
#Keep only relevant columns
total_income_df = data['Total annual income'][['MSOA code', 'Total annual income (£)']]
net_income_df = data['Net annual income'][['MSOA code', 'Net annual income (£)']]
net_before_housing_df = data['Net income before housing costs'][['MSOA code', 'Net annual income before housing costs (£)']]
net_after_housing_df = data['Net income after housing costs'][['MSOA code', 'Net annual income after housing costs (£)']]

# Merge all DataFrames on 'MSOA code'
merged_df = total_income_df \
    .merge(net_income_df, on='MSOA code') \
    .merge(net_before_housing_df, on='MSOA code') \
    .merge(net_after_housing_df, on='MSOA code')

# Rename the columns
merged_df = merged_df.rename(columns={
    'MSOA code': 'msoa11cd',
    'Total annual income (£)': 'total_annual_income',
    'Net annual income (£)': 'net_annual_income',
    'Net annual income before housing costs (£)': 'net_annual_income_before_housing_costs',
    'Net annual income after housing costs (£)': 'net_annual_income_after_housing_costs'
})

# Display the renamed DataFrame
merged_df.head()

,msoa11cd,total_annual_income,net_annual_income,net_annual_income_before_housing_costs,net_annual_income_after_housing_costs
0,E02004297,41400,30500,28800,27700
1,E02004290,41100,31200,30100,28600
2,E02004298,44300,33800,31300,31400
3,E02004299,35400,29500,27600,25000
4,E02004291,34500,27200,26500,23800


## 4. Load GIS shapefile 

In [9]:
# Attempt to read from the server first, fallback to local path if it fails
msoa2011boundary_df = gpd.read_file(msoa2011_shapefile)
print("Shapefile loaded successfully from the server.")

# Display the first few rows
msoa2011boundary_df.head()

Shapefile loaded successfully from the server.


,MSOA11CD,MSOA11NM,BNG_E,BNG_N,LONG_,LAT,Shape_Leng,GlobalID,geometry
0,E02000001,City of London 001,532384,181355,-0.093490,51.5156,9651.221144,ae98212b-e51e-43cb-8eb8-9a9edc228ea7,"POLYGON ((532153.703 182165.155, 532158.250 18..."
1,E02000002,Barking and Dagenham 001,548267,189685,0.138756,51.5865,8306.888230,6cc3f460-0089-4d6f-85ef-c05f4e78d547,"POLYGON ((548881.304 190819.980, 548881.125 19..."
2,E02000003,Barking and Dagenham 002,548259,188520,0.138149,51.5760,9359.512342,38e8a480-1669-4ca6-a07b-b628b1c7c302,"POLYGON ((548958.555 189072.176, 548954.517 18..."
3,E02000004,Barking and Dagenham 003,551004,186412,0.176828,51.5564,8475.840309,232da89a-870b-44f8-a2e5-d7dc5ad28f0f,"POLYGON ((551550.056 187364.705, 551528.633 18..."
4,E02000005,Barking and Dagenham 004,548733,186824,0.144267,51.5607,7321.657104,3926d87b-f86e-44c9-88a1-0c49e220a56f,"POLYGON ((549237.051 187627.941, 549241.319 18..."


## 5. GIS data management

### Remove Rename fields

In [10]:
#Drop and rename fields
msoa2011boundary_df.drop(['Shape_Leng', 'GlobalID','BNG_E','BNG_N','LONG_','LAT'], 1, inplace = True)
msoa2011boundary_df.rename(columns = {'MSOA11CD':'msoa11cd','MSOA11NM':'msoa11nm'}, inplace = True)
msoa2011boundary_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_13256\931186528.py:2: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  msoa2011boundary_df.drop(['Shape_Leng', 'GlobalID','BNG_E','BNG_N','LONG_','LAT'], 1, inplace = True)


,msoa11cd,msoa11nm,geometry
0,E02000001,City of London 001,"POLYGON ((532153.703 182165.155, 532158.250 18..."
1,E02000002,Barking and Dagenham 001,"POLYGON ((548881.304 190819.980, 548881.125 19..."
2,E02000003,Barking and Dagenham 002,"POLYGON ((548958.555 189072.176, 548954.517 18..."
3,E02000004,Barking and Dagenham 003,"POLYGON ((551550.056 187364.705, 551528.633 18..."
4,E02000005,Barking and Dagenham 004,"POLYGON ((549237.051 187627.941, 549241.319 18..."


### Combine with boundary lookup table

#### LSOA21 to MSOA21 to LAD22

In [11]:
#MSOA11 LOOKUP
msoa11_lookup_df = pd.read_excel(open(r"N:\Geodatabase\Raw_Data\Look up tables\MSOA_(2011)_to_MSOA_(2021)_to_Local_Authority_District_(2022)_Lookup_for_England_and_Wales_7815823766405070629(3).xlsx", 'rb'), sheet_name='MSOA_(2011)_to_MSOA_(2021)__0') 

columns_to_keep = [
    'MSOA11CD','LAD22CD','LAD22NM'
]

#drop unwanted fields
msoa11_lookup_cleaned_df = msoa11_lookup_df[columns_to_keep]

#rename msoa11cd to make merging cleaner
msoa11_lookup_cleaned_df.rename(columns={
    'MSOA11CD': 'msoa11cd',
    'LAD22CD': 'lad22cd',
    'LAD22NM': 'lad22nm',
}, inplace=True)

msoa11_lookup_cleaned_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_13256\2762448710.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  msoa11_lookup_cleaned_df.rename(columns={


,msoa11cd,lad22cd,lad22nm
0,E02000001,E09000001,City of London
1,E02000002,E09000002,Barking and Dagenham
2,E02000003,E09000002,Barking and Dagenham
3,E02000004,E09000002,Barking and Dagenham
4,E02000005,E09000002,Barking and Dagenham


In [12]:
msoa2011boundary_df = pd.merge(msoa2011boundary_df, msoa11_lookup_cleaned_df, how = 'left', on = 'msoa11cd')
msoa2011boundary_df.head()

,msoa11cd,msoa11nm,geometry,lad22cd,lad22nm
0,E02000001,City of London 001,"POLYGON ((532153.703 182165.155, 532158.250 18...",E09000001,City of London
1,E02000002,Barking and Dagenham 001,"POLYGON ((548881.304 190819.980, 548881.125 19...",E09000002,Barking and Dagenham
2,E02000003,Barking and Dagenham 002,"POLYGON ((548958.555 189072.176, 548954.517 18...",E09000002,Barking and Dagenham
3,E02000004,Barking and Dagenham 003,"POLYGON ((551550.056 187364.705, 551528.633 18...",E09000002,Barking and Dagenham
4,E02000005,Barking and Dagenham 004,"POLYGON ((549237.051 187627.941, 549241.319 18...",E09000002,Barking and Dagenham


#### LAD22 to REGION22

In [13]:
# Define the file path for lad to region
file_path = r"N:\Geodatabase\Raw_Data\Look up tables\Local_Authority_District_to_Region_(December_2022)_Lookup_in_England.csv"

# Read the Excel file
ladregionlookup_df = pd.read_csv(file_path)  # Reads the first sheet by default

#drop unwanted fields
ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)

# Rename fields
ladregionlookup_df.rename(
    columns={
        'LAD22CD': 'lad22cd',
        'RGN22CD': 'rgn22cd',
        'RGN22NM': 'rgn22nm'
       }, 
    inplace=True
)

# Display the first few rows
ladregionlookup_df.head()

C:\Users\abhimanya.achara\AppData\Local\Temp\ipykernel_13256\2710224882.py:8: FutureWarning: In a future version of pandas all arguments of DataFrame.drop except for the argument 'labels' will be keyword-only.
  ladregionlookup_df.drop(['LAD22NM','ObjectId'],1,inplace = True)


,lad22cd,rgn22cd,rgn22nm
0,E06000001,E12000001,North East
1,E06000002,E12000001,North East
2,E06000003,E12000001,North East
3,E06000004,E12000001,North East
4,E06000005,E12000001,North East


In [14]:
msoa2011boundary_df = pd.merge(msoa2011boundary_df, ladregionlookup_df, how = 'left', on = 'lad22cd')
msoa2011boundary_df.head()

,msoa11cd,msoa11nm,geometry,lad22cd,lad22nm,rgn22cd,rgn22nm
0,E02000001,City of London 001,"POLYGON ((532153.703 182165.155, 532158.250 18...",E09000001,City of London,E12000007,London
1,E02000002,Barking and Dagenham 001,"POLYGON ((548881.304 190819.980, 548881.125 19...",E09000002,Barking and Dagenham,E12000007,London
2,E02000003,Barking and Dagenham 002,"POLYGON ((548958.555 189072.176, 548954.517 18...",E09000002,Barking and Dagenham,E12000007,London
3,E02000004,Barking and Dagenham 003,"POLYGON ((551550.056 187364.705, 551528.633 18...",E09000002,Barking and Dagenham,E12000007,London
4,E02000005,Barking and Dagenham 004,"POLYGON ((549237.051 187627.941, 549241.319 18...",E09000002,Barking and Dagenham,E12000007,London


### Add data management fields

In [16]:
# Add new data management fields with specified values
msoa2011boundary_df['data_source'] = "ONS"
msoa2011boundary_df['data_resolution'] = "MSOA 2011"
msoa2011boundary_df['data_time_period'] = "2020"
msoa2011boundary_df['data_web_link'] = "https://www.ons.gov.uk/employmentandlabourmarket/peopleinwork/earningsandworkinghours/datasets/smallareaincomeestimatesformiddlelayersuperoutputareasenglandandwales"
msoa2011boundary_df.head()

,msoa11cd,msoa11nm,geometry,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link
0,E02000001,City of London 001,"POLYGON ((532153.703 182165.155, 532158.250 18...",E09000001,City of London,E12000007,London,ONS,MSOA 2011,2020,https://www.ons.gov.uk/employmentandlabourmark...
1,E02000002,Barking and Dagenham 001,"POLYGON ((548881.304 190819.980, 548881.125 19...",E09000002,Barking and Dagenham,E12000007,London,ONS,MSOA 2011,2020,https://www.ons.gov.uk/employmentandlabourmark...
2,E02000003,Barking and Dagenham 002,"POLYGON ((548958.555 189072.176, 548954.517 18...",E09000002,Barking and Dagenham,E12000007,London,ONS,MSOA 2011,2020,https://www.ons.gov.uk/employmentandlabourmark...
3,E02000004,Barking and Dagenham 003,"POLYGON ((551550.056 187364.705, 551528.633 18...",E09000002,Barking and Dagenham,E12000007,London,ONS,MSOA 2011,2020,https://www.ons.gov.uk/employmentandlabourmark...
4,E02000005,Barking and Dagenham 004,"POLYGON ((549237.051 187627.941, 549241.319 18...",E09000002,Barking and Dagenham,E12000007,London,ONS,MSOA 2011,2020,https://www.ons.gov.uk/employmentandlabourmark...


### Update Area

In [17]:
#Update Area in Hectares
# Ensure the dataframe has a valid geometry column
if 'geometry' in msoa2011boundary_df.columns:
    # Calculate the area in square meters and convert to hectares (1 hectare = 10,000 m²)
    msoa2011boundary_df['area_ha'] = msoa2011boundary_df.geometry.area / 10_000

    # Print confirmation message
    print("Field 'area_ha' has been added and updated successfully.")
else:
    print("Error: No 'geometry' column found in msoa2011boundary_df. Ensure it's a GeoDataFrame with valid geometries.")
    
msoa2011boundary_df.head()

Field 'area_ha' has been added and updated successfully.


,msoa11cd,msoa11nm,geometry,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha
0,E02000001,City of London 001,"POLYGON ((532153.703 182165.155, 532158.250 18...",E09000001,City of London,E12000007,London,ONS,MSOA 2011,2020,https://www.ons.gov.uk/employmentandlabourmark...,315.147533
1,E02000002,Barking and Dagenham 001,"POLYGON ((548881.304 190819.980, 548881.125 19...",E09000002,Barking and Dagenham,E12000007,London,ONS,MSOA 2011,2020,https://www.ons.gov.uk/employmentandlabourmark...,216.156066
2,E02000003,Barking and Dagenham 002,"POLYGON ((548958.555 189072.176, 548954.517 18...",E09000002,Barking and Dagenham,E12000007,London,ONS,MSOA 2011,2020,https://www.ons.gov.uk/employmentandlabourmark...,214.151524
3,E02000004,Barking and Dagenham 003,"POLYGON ((551550.056 187364.705, 551528.633 18...",E09000002,Barking and Dagenham,E12000007,London,ONS,MSOA 2011,2020,https://www.ons.gov.uk/employmentandlabourmark...,249.294610
4,E02000005,Barking and Dagenham 004,"POLYGON ((549237.051 187627.941, 549241.319 18...",E09000002,Barking and Dagenham,E12000007,London,ONS,MSOA 2011,2020,https://www.ons.gov.uk/employmentandlabourmark...,118.795417


# 7. Combine Geometry and data

In [18]:
ohid_income_msoa2011_gdb_df = pd.merge(msoa2011boundary_df, merged_df, how = 'left', on='msoa11cd')
ohid_income_msoa2011_gdb_df.head()

,msoa11cd,msoa11nm,geometry,lad22cd,lad22nm,rgn22cd,rgn22nm,data_source,data_resolution,data_time_period,data_web_link,area_ha,total_annual_income,net_annual_income,net_annual_income_before_housing_costs,net_annual_income_after_housing_costs
0,E02000001,City of London 001,"POLYGON ((532153.703 182165.155, 532158.250 18...",E09000001,City of London,E12000007,London,ONS,MSOA 2011,2020,https://www.ons.gov.uk/employmentandlabourmark...,315.147533,101800,62700,65000,59300
1,E02000002,Barking and Dagenham 001,"POLYGON ((548881.304 190819.980, 548881.125 19...",E09000002,Barking and Dagenham,E12000007,London,ONS,MSOA 2011,2020,https://www.ons.gov.uk/employmentandlabourmark...,216.156066,43600,32500,28600,22500
2,E02000003,Barking and Dagenham 002,"POLYGON ((548958.555 189072.176, 548954.517 18...",E09000002,Barking and Dagenham,E12000007,London,ONS,MSOA 2011,2020,https://www.ons.gov.uk/employmentandlabourmark...,214.151524,50900,39200,33000,27700
3,E02000004,Barking and Dagenham 003,"POLYGON ((551550.056 187364.705, 551528.633 18...",E09000002,Barking and Dagenham,E12000007,London,ONS,MSOA 2011,2020,https://www.ons.gov.uk/employmentandlabourmark...,249.294610,52000,38900,34300,29300
4,E02000005,Barking and Dagenham 004,"POLYGON ((549237.051 187627.941, 549241.319 18...",E09000002,Barking and Dagenham,E12000007,London,ONS,MSOA 2011,2020,https://www.ons.gov.uk/employmentandlabourmark...,118.795417,48100,35200,30000,25800


In [19]:
ohid_income_msoa2011_gdb_df = ohid_income_msoa2011_gdb_df.drop_duplicates(subset='msoa11cd', keep='first')

# 9. Upload to geodatabase

In [20]:
# Define database connection parameters
db_host = "PRIORPSRV03"
db_name = "gis"
db_port = "5432"
db_schema = "uk_new"
table_name = "ons_masoa2011_income_2020"  # Desired table name
primary_key_column = "msoa11cd"  # Define the primary key column (change based on your dataset)
geometry_column = "geometry"  # Default geometry column
# Create the database connection string (Windows Authentication - Trusted Connection)
conn_str = f"postgresql+psycopg2://@{db_host}:{db_port}/{db_name}?sslmode=disable"

# Create a SQLAlchemy engine
engine = create_engine(conn_str)

In [21]:
# Ensure the GeoDataFrame has a valid CRS before writing
if ohid_income_msoa2011_gdb_df.crs is None:
    print("Warning: GeoDataFrame has no CRS. Setting default to EPSG:27700 (British National Grid).")
    ohid_income_msoa2011_gdb_df.set_crs(epsg=27700, inplace=True)

In [22]:
# Publish the GeoDataFrame to PostGIS
ohid_income_msoa2011_gdb_df.to_postgis(
    name=table_name,
    con=engine,
    schema=db_schema,
    if_exists="replace",
    index=False,
    dtype={'geometry': 'POLYGON'}  # Ensure geometry is stored as MULTIPOLYGON
)

print(f"Data successfully uploaded to PostGIS: {db_schema}.{table_name}")

# Connect to the database to modify table structure
with engine.connect() as conn:
    # Set Primary Key (if it doesn't exist already)
    alter_pk_query = text(f"""
        ALTER TABLE {db_schema}.{table_name}
        ADD CONSTRAINT {table_name}_pkey PRIMARY KEY ({primary_key_column});
    """)
    
    # Create Spatial Index
    create_spatial_index_query = text(f"""
        CREATE INDEX {table_name}_geom_idx
        ON {db_schema}.{table_name}
        USING GIST ({geometry_column});
    """)

    try:
        conn.execute(alter_pk_query)  # Add Primary Key
        print(f"Primary key set on column: {primary_key_column}")
    except Exception as e:
        print(f"Could not set primary key. It may already exist. Error: {e}")

    try:
        conn.execute(create_spatial_index_query)  # Add Spatial Index
        print(f"Spatial index created for geometry column: {geometry_column}")
    except Exception as e:
        print(f"Could not create spatial index. It may already exist. Error: {e}")

print(f"GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: {db_schema}.{table_name}")

Data successfully uploaded to PostGIS: uk_new.ons_masoa2011_income_2020
Primary key set on column: msoa11cd
Spatial index created for geometry column: geometry
GeoDataFrame successfully published to PostGIS with Primary Key and Spatial Index: uk_new.ons_masoa2011_income_2020
